In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-huggingface \
    faiss-cpu \
    sentence-transformers \
    pypdf \
    groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2

In [4]:
import urllib.request
from pypdf import PdfReader
import io

# Download the PDF into memory
url = "https://arxiv.org/pdf/1706.03762"
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
pdf_bytes = urllib.request.urlopen(req).read()

# Extract text from all pages
reader = PdfReader(io.BytesIO(pdf_bytes))
pages = [page.extract_text() for page in reader.pages]
full_text = "\n".join(pages)

# Sanity check
print(f"Pages loaded: {len(pages)}")
print(f"Total characters: {len(full_text)}")
print("\n--- First 500 characters ---")
print(full_text[:500])

Pages loaded: 15
Total characters: 39629

--- First 500 characters ---
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_text(full_text)

# Sanity check
print(f"Total chunks: {len(chunks)}")
print(f"Avg chunk size: {sum(len(c) for c in chunks) // len(chunks)} characters")
print("\n--- Chunk 0 ---")
print(chunks[0])
print("\n--- Chunk 1 ---")
print(chunks[1])
print("\nEnd of chunk 0:", chunks[0][-60:])
print("Start of chunk 1:", chunks[1][:60])

Total chunks: 89
Avg chunk size: 455 characters

--- Chunk 0 ---
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu

--- Chunk 1 ---
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser ∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network 

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

# Load the embedding model (downloads ~90MB on first run, cached after)
embedder = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Embed a single chunk first so you can SEE what a vector looks like
sample_vector = embedder.embed_query(chunks[0])

print(f"Embedding model loaded!")
print(f"Vector size: {len(sample_vector)} dimensions")
print(f"\nFirst 10 values of chunk 0's vector:")
print([round(v, 4) for v in sample_vector[:10]])
print("\nThis is what 'meaning' looks like as numbers.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!
Vector size: 384 dimensions

First 10 values of chunk 0's vector:
[0.1132, -0.0085, 0.0096, 0.0503, 0.008, -0.0028, -0.0367, -0.001, -0.0289, 0.1407]

This is what 'meaning' looks like as numbers.


In [7]:
from langchain_community.vectorstores import FAISS

# Embed all 89 chunks and build the index (takes ~10-20 seconds)
print("Building FAISS index...")
db = FAISS.from_texts(chunks, embedder)

print(f"Index built!")
print(f"Vectors stored: {db.index.ntotal}")

# Test it immediately — search for something
test_results = db.similarity_search("What is the attention mechanism?", k=3)

print(f"\n--- Top 3 chunks for: 'What is the attention mechanism?' ---")
for i, doc in enumerate(test_results):
    print(f"\n[{i+1}] {doc.page_content[:200]}...")

Building FAISS index...
Index built!
Vectors stored: 89

--- Top 3 chunks for: 'What is the attention mechanism?' ---

[1] reduced to a constant number of operations, albeit at the cost of reduced effective resolution due
to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as
descr...

[2] from our models and present and discuss examples in the appendix. Not only do individual attention
heads clearly learn to perform different tasks, many appear to exhibit behavior related to the syntac...

[3] 3.2.3 Applications of Attention in our Model
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder laye...


In [9]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("GROQ_API_KEY")

In [10]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
GROQ_API_KEY = secrets.get_secret("GROQ_API_KEY")
print("API key loaded!")

API key loaded!


In [16]:
from groq import Groq

# 1. Your question
query = "What optimizer did they use for training?"

# 2. Retrieve the top 3 most relevant chunks
results = db.similarity_search(query, k=5)
context = "\n\n".join(doc.page_content for doc in results)

# 3. Build the prompt
system_prompt = """You are a helpful assistant that answers questions about a research paper.
Answer ONLY using the context provided below. If the answer is not in the context, say "I don't find that in the paper."
Do not make anything up.

Context:
""" + context

# 4. Send to Groq
client = Groq(api_key=GROQ_API_KEY)
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query}
    ]
)

answer = response.choices[0].message.content

# 5. Print everything so you can see each step
print("=== QUERY ===")
print(query)
print("\n=== RETRIEVED CONTEXT (what we sent to the LLM) ===")
print(context[:800], "...")
print("\n=== ANSWER ===")
print(answer)

=== QUERY ===
What optimizer did they use for training?

=== RETRIEVED CONTEXT (what we sent to the LLM) ===
(3.5 days).
5.3 Optimizer
We used the Adam optimizer [20] with β1 = 0.9, β2 = 0.98 and ϵ = 10 −9. We varied the learning
rate over the course of training, according to the formula:
lrate = d−0.5
model · min(step_num−0.5, step_num · warmup_steps−1.5) (3)
This corresponds to increasing the learning rate linearly for the first warmup_steps training steps,
and decreasing it thereafter proportionally to the inverse square root of the step number. We used
warmup_steps = 4000.
5.4 Regularization

warmup_steps = 4000.
5.4 Regularization
We employ three types of regularization during training:
7
Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models on the
English-to-German and English-to-French newstest2014 tests at a fraction of the training cost.
Model
BLEU Tr ...

=== ANSWER ===
They used the Adam optimizer with β1 = 0.9, β2 = 0.98, and ϵ = 10^−9.
